# Installing and importing Python packages in the Pyodide kernel

[jupyterlite-pyodide-kernel](https://github.com/jupyterlite/pyodide-kernel), powered by [Pyodide](https://pyodide.org), can install several kinds of Python packages, as well as WASM shims for binary libraries.

- Pyodide binary libraries
  - special `.zip` files of _binary libraries_ compiled to WASM
- Python wheels
  - _"noarch" wheels_, ending in `py3-none-any.whl`
  - _"pyemscripten" wheels_, conforming to [PEP-783](https://peps.python.org/pep-0783), e.g. `cp3XX-cp3XX-pyemscripten_YYYY_Z_wasm32.whl`

## Pyodide-compatible package sources

Packages can be feched from a number of sources, including a site's Pyodide distribution, _package indexes_ like [PyPI](https://pypi.org),
or even inside a JupyterLite site.

The available sources, and how interactive content must be written to use them, depends on how a JupyterLite site is built.

- Some binary libraries and known-compatible wheels are described in a Pyodide distribution's `pyodide-lock.json`
  - these usually do not need to be explicitly installed, and are ready to `import`
- `jupyterlite-pyodide-kernel` includes its own "noarch" wheel for the runtime kernel
  - this includes some compatibility shim wheels for the Jupyter stack such as `ipykernel`
- Site owner-chosen wheels deployed with a JupyterLite site:
  - downloaded from a package index and added to `pyodide-lock.json` with `jupyterlite-pyodide-kernel[lock]`
  - added to a small local snapshot of the PyPI Warehouse JSON API
- "noarch" and "pyemscripten" wheels on a remote package index
- "noarch" wheels built within a kernel session

## `PyodideLockAddon` (build time)

With `jupyterlite-pyodide-kernel[lock]` installed, a site owner can extend their site's Pyodide distribution with extra packages.

This approach allows for:

- writing interactive content without the `%pip` magic, as all locked packages are ready to `import`
- ensuring versions of known packages don't change after a site is deployed
- co-hosting any required wheels, reducing runtime installation complexity, avoiding many HTTP requests for metadata and CORS pitfalls

> See JupyterLite's own [`jupyter_lite_config.json`](https://github.com/jupyterlite/jupyterlite/blob/main/examples/jupyter_lite_config.json)
> for a full example of managing complex dependencies by:
>
> - describing specs in a project's `pyproject.toml#/dependency-groups`
> - capturing constraints for future site builds
> - patching runtime dependencies
> - delegating storage to third-party CDNs

### Example: Including a locally-built wheel in a built site

Consider a Python project which builds a JupyterLite site as a demo for a locally-built, "noarch" wheel. 

> This approach works particularly well with short-term hosting such as:
> 
> - [ReadTheDocs pull request previews](https://docs.readthedocs.com/platform/latest/pull-requests.html)
> - [GitLab Pages parallel deployments](https://docs.gitlab.com/user/project/pages/parallel_deployments)

<details><summary>Show project structure...</summary>

```bash
./
    pyproject.toml
    src/
        my_package
    demo/
        jupyter_lite_config.json
        my-package-demo.ipynb
    # ... after `pyproject-build`
    dist/
        my_package-0.0.1-py3-none-any.whl
    # ... after `cd demo && jupyter lite build`
    build/
        demo/
            static/
                pyodide-lock/
                    pyodide-lock.json
                    my_package-0.0.1-py3-none-any.whl
```

</details>

<details><summary>Show <code>demo/jupyter_lite_config.json</code>...</summary>

```json
{
    "LiteBuildConfig": {
        "contents": ["."],
        "output_dir": "../build/demo",
    },
    "PyodideLockAddon": {
        "enabled": true,
        "specs": ["ipywidgets"],
        "wheels": ["../dist"]
    }
}
```

</details>

After building the wheel, running `jupyter lite build` in `demo/` will:

- use `uv` to solve an environment constrained by prioritizing:
  - any compatible local `wheels` in `dist/` and their dependencies
  - any added [PEP-508](https://peps.python.org/pep-0508) `specs` and their dependencies
  - the upstream `pyodide-lock.json`
- cache any remote wheels to discover importable names
- generate a new `pyodide-lock.json`
- copy the generated lock file to `build/demo/static/pyodide-lock/`
  - optionally copy remote wheels
- update the runtime `build/demo/jupyter-config.json` to use the new lock file

When a user visits the built site, `import my_package` will work as per a "normal" IPython session.

## `piplite` (run time)

`piplite` is a wrapper around `micropip` allowing for:

- installing the runtime `pyodide_kernel` and its dependencies such as `ipython`
- overriding IPyhton's `%pip` magic to install packages not included in the Pyodide distribution

### `%pip install -q`

For packages not covered in the techniques above, the recommended way to ensure a package is installed is to use the `%pip` magic:

In [ ]:
%pip install -q "traitlets >=5" ipython


- `%pip` helps keep your notebooks **portable** between different IPython runtimes.
- `-q` helps keep this "setup" step reasonable when run under `ipykernel`
- putting `%pip` calls together in a single cell, with no other executable code, helps isolate dependency issues

`%pip` supports a number of additional options:

In [ ]:
%pip --help

#### `--requirements`

In addition to (or instead of) package names, any number of `--requirements` (or
shorthand `-r`) may be given, pointing to [requirements files][reqs-txt].

[reqs-txt]: https://pip.pypa.io/en/stable/reference/requirements-file-format

These can be relative or absolute (though this can be tricky to predict).

In [ ]:
%pip install -r data/requirements.txt -r ../data/more-requirements.txt

> Using a `requirements.txt` is a good way to separate your content from your environment,
> and could be combined with other techniques to make even more portable notebooks.

##### `requirement.txt` lines

A number of simple types of requirement specifications are
[supported](./data/requirements.txt), including:

- package names
- package names with version specifiers supported by `micropip`
- URLs of `.whl` archives
- `-r` to _more_ requirements files
  - relative to the _requiring_ file, not the `%pip` working directory

Some known **unsupported** features:

- `.` or other local in-development paths
  - but you _could_
    - `%pip install` a build tool (e.g. `flit`)
      - use its Python API
        - install the resulting `.whl`
- `--editable` (or `-e.`) local or remote paths
- any version control system (VCS) paths
- non-`.whl` URLs or local archives
- `--constraint` (or `-c`) constraint files

### Dealing with (missing) dependencies

Even if a PyPI package _would be_ installable in the Pyodide kernel, sometimes its
dependencies won't be.

In [ ]:
try:
    %pip install jupyter_server
except Exception as err:
    print(err)

#### `--verbose`

As the real `pip` doesn't have an equivalent, `%pip` in the Pyodide kernel maps the
`--verbose` flag to `keep_going`.

In [ ]:
try:
    %pip install --verbose jupyter_server
except Exception as err:
    print(err)

> Leaving this on for real `pip` generates a **lot** of output!

#### `--no-deps`

If some missing dependencies don't bother you, you can forge ahead without _any_
dependencies with the `--no-deps` flag.

In [ ]:
%pip install jupyter_server --no-deps
import jupyter_server

jupyter_server.__version__

While importable, it won't have all of its features, and may require special approaches
to access features.

In [ ]:
try:
    import jupyter_server.services.contents.filemanager
except Exception as err:
    print(err)

> Going down this road can be long, depending on how much you really need a particular
function.

### The Hard Way

The `piplite` package is importable, and can be used directly.

#### Importing `piplite`

`piplite` needs to be imported before it is used.

In [ ]:
import piplite

This package **won't** be installable in a "traditional" IPython installation, so you
can gate it with an import check:

In [ ]:
try:
    import piplite
except ImportError:
    piplite = None

#### `piplite.install`

> `piplite.install` is a wrapper around
> [`micropip.install`](https://pyodide.org/en/stable/usage/loading-packages.html#micropip),
> and offers more browser-focused options than `%pip`

> **NOTE** Due to browser limitations, `piplite.install` is an _asynchronous` function, so it must be `await`ed.

`piplite.install` supports either a single package:

In [ ]:
await piplite.install("traitlets")

or a list of packages:

In [ ]:
await piplite.install(["traitlets", "IPython"])

It also has many additional options:

In [ ]:
?piplite.install